# Introduction to Large Language Models (LLMs)

Large Language Models (LLMs) have revolutionized natural language processing (NLP) and have become foundational tools in various AI applications. These models, trained on massive datasets, are capable of generating human-like text, translating languages, summarizing documents, answering questions, and much more. In this notebook, we will explore how to use large language models, particularly those built on architectures like GPT, BERT, and LLaMA.

## What Are Large Language Models?

LLMs are a type of deep learning model specifically designed to understand and generate text. They are typically based on transformer architectures, which allow them to process and generate text in parallel, making them highly efficient and powerful. The "large" in LLMs refers to the vast number of parameters—often in the billions—these models possess, which enables them to capture the nuances and complexities of human language.

### Key Features of LLMs:

- **Contextual Understanding:** LLMs can generate text that is contextually relevant, meaning they understand the context in which words are used, leading to more accurate and coherent outputs.
- **Transfer Learning:** These models can be fine-tuned on specific tasks with smaller datasets, making them highly versatile for various NLP applications.
- **Generative Capabilities:** Beyond understanding text, LLMs can generate creative and complex content, from poetry to technical explanations.

## Why Use LLMs?

The widespread adoption of LLMs is driven by their ability to perform a wide range of tasks with minimal human intervention. They are used in chatbots, content creation, sentiment analysis, language translation, code generation, and more. Their ability to generalize across tasks without needing task-specific models has made them invaluable in both industry and research.

## What Will You Learn?

In this notebook, you will learn how to leverage open-source LLMs for text generation and other NLP tasks using Python and popular libraries such as `transformers`. We will guide you through:
- Setting up the environment for working with LLMs.
- Loading pre-trained models and tokenizers.
- Generating text based on specific prompts.

By the end of this notebook, you will have a solid understanding of how to implement and use LLMs in practical scenarios, opening the door to countless AI-powered applications.

Let’s get started!


In [ ]:
!pip install accelerate sentence-transformers faiss-cpu

# LLM: Qwen2

- [Model details](https://qwenlm.github.io/blog/qwen2/)
- Completely open, you do not have to apply for access
- We will use a quantized version of the model (preserves accuracy of the model while significantly reduces memory requirements). If you are interested, read more [here](https://huggingface.co/docs/optimum/concept_guides/quantization).

In [ ]:
from transformers import pipeline
import torch

device = "cuda" # the device to load the model onto
model_id = "/models/GaMS3-12B-Instruct"

model = pipeline(
    "text-generation",
    model=model_id,
    device_map=device
)

## Response generation function

In [ ]:
# A function that returns the answer from GaMS3 model using pipeline
def gams3_generate(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]

    # We get the response with only one call
    # However, the response is the conversation list, so we have to extract the content from the last message
    response = model(messages, max_new_tokens=1024)

    return response[0]["generated_text"][-1]["content"]

In [ ]:
# Chat about anything
prompt = "V 3 stavkih mi na kratko predstavi veliki jezikovni model."
response = gams3_generate(prompt)
print(response)

# RAG (Retrieval-augmented Generation)

[![RAG.jpg](https://i.postimg.cc/W1V5BJWn/RAG.jpg)](https://postimg.cc/Mvs0RX8M)

**In simple terms, RAG is to LLMs what an open-book exam is to humans.**

The concept of an open-book exam centers around assessing a student's reasoning abilities rather than their capacity to memorize specific details. In a similar vein, RAG separates factual knowledge from the LLM’s reasoning capabilities. This factual information is stored in an external knowledge source, which is both easily accessible and updatable:

- **Parametric knowledge:** Knowledge that is learned during training and implicitly stored within the neural network's weights.
- **Non-parametric knowledge:** Information that is stored externally, for example, in a vector database.

The RAG workflow consists of:

1. **The Retrieve**: The user query is used to retrieve relevant context from an external knowledge source. For this, the user query is embedded using an embedding model into the same vector space as the additional context in the vector database. This enables a similarity search, and the top k closest data objects from the vector database are returned.
2. **Augment**: The user query and the retrieved additional context are incorporated into a prompt template.
3. **Generate**: Finally, the retrieval-augmented prompt is fed to the LLM.

## Vector database

In [ ]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from sentence_transformers import SentenceTransformer, util
import faiss

# Sample internal documents that represent the knowledge base
corpus = [
    "Pravilnik o delu na daljavo (v2024): Zaposleni z manj kot 12 meseci delovne dobe v podjetju SloTech Inovacije morajo biti v pisarni vsaj 3 dni v tednu. Senior razvijalci imajo možnost popolnega dela na daljavo ob odobritvi vodje ekipe.",
    "Projekt Triglav - Dokumentacija za razvijalce: Za postavitev lokalnega okolja sledite tem korakom: 1. Klunirajte repozitorij, 2. Zaženite ukaz 'docker-compose up -d', 3. Uvozite testno bazo preko skripte 'scripts/init_db.sh'.",
    "Varnostni priročnik IT: API ključi strank se nikoli ne smejo shranjevati v datotekah .env ali neposredno v kodi. Obvezna je uporaba internega HashiCorp Vault sistema za vse produkcijske skrivnosti.",
    "Vizija podjetja 2026: SloTech Inovacije stremi k postati vodilni ponudnik rešitev na področju lokalnih LLM modelov v regiji Adriatik, s poudarkom na zasebnosti in varnosti podatkov.",
    "Navodila za potne stroške: Dnevnica za službena potovanja znotraj Slovenije znaša 23,00 EUR, če potovanje traja nad 12 ur. Zahtevki se oddajajo preko aplikacije MojaPlača do 5. v mesecu.",
    "Onboarding vodnik: Novi sodelavci dobijo dostop do Slack kanalov #general, #it-podpora in #projekti-obvestila prvi delovni dan. Dostop do Jira sistema uredi sistemski administrator po opravljenem varnostnem izobraževanju.",
    "Tehnični standardi: V podjetju uporabljamo izključno Python 3.10 ali novejši za vse nove mikrostoritve. Dokumentacija mora biti napisana v angleškem jeziku v formatu Markdown.",
    "Projekt Triglav - Opis: Projekt Triglav je mobilna aplikacija za gorsko reševanje, ki uporablja satelitske podatke za lociranje pohodnikov v realnem času brez uporabe mobilnega signala.",
    "Kodeks obnašanja: V SloTech Inovacije spodbujamo kulturo odprte komunikacije, spoštovanja in nenehnega učenja. Vsakršna oblika diskriminacije vodi v takojšen disciplinski postopek.",
    "Navodila za uporabo internega orodja SloBot: SloBot je interno AI orodje, ki temelji na Llama-3 modelu. Uporablja se lahko za povzemanje sestankov, ne sme pa se vanj vpisovati osebnih podatkov strank."
]

# Initialize retriever model using a newer sentence-transformer model
retriever_model = SentenceTransformer('/models/bge-m3')

# Encode corpus
corpus_embeddings = retriever_model.encode(corpus, convert_to_tensor=True)

The result of sending the corpus embedding through the encoder is the following set of vectors

In [ ]:
corpus_embeddings

We can check the matrix dimensions using the following command. We get 10 1024-dimensional vectors.

In [ ]:
corpus_embeddings.shape

Our next step is to convert our vectors in a searchable vector databe. For this, we will use FAISS index, which is simple to use and works on CPU.

In [ ]:
# This code initializes a FAISS index for efficient similarity search over a collection of vector embeddings.
corpus_embeddings_np = corpus_embeddings.cpu().numpy()
index = faiss.IndexFlatL2(corpus_embeddings_np.shape[1])
index.add(corpus_embeddings_np)

## RAG Inference

Let's start by defining the same set of prompts we used in our previous excercise. This time we will provide an additional context to the model using RAG.

In [ ]:
prompts = [
    "Napiši kratek Python primer za uporabo knjižnice pandas.",
    "Kakšen je postopek za delo od doma za zaposlene z manj kot letom dni delovne dobe?",
    "Katere korake moram izvesti za postavitev razvojnega okolja pri projektu Triglav?",
    "Pripravi osnutek objave za LinkedIn, v kateri podjetje napoveduje novo sodelovanje na področju umetne inteligence.",
    "Kakšni so varnostni protokoli glede shranjevanja API ključev strank v našem podjetju?"
]

### Simple prompt

The simplest way to use RAG is just to append context to the prompt. We will first retrieve the most relevant document from our base and append it to the instruction.

In [ ]:
def simple_rag_query(query):
    # Step 1: Retrieve
    # Encode query and retrieve the most relevant document
    query_embedding = retriever_model.encode(query, convert_to_tensor=True)
    query_embedding_np = query_embedding.cpu().numpy()
    query_embedding_np = query_embedding_np.reshape(1, -1)
    distances, top_k_index = index.search(query_embedding_np, k=1)
    print('Index of the most relevant document:', top_k_index)

    # Fetch the most relevant document
    doc_index = top_k_index[0]
    retrieved_doc = corpus[doc_index[0]]

    # Step 2: Augment
    # Combine query and retrieved docs for generation input
    combined_input = query + "\n\n" + retrieved_doc
    print('Augmented prompt:', combined_input)

    # Step 3: Generate response using the T5 model
    response = gams3_generate(combined_input)

    return response

In [ ]:
# Test the RAG implementation
for query in prompts:
    print(30*"-")
    print("Query:", query)
    response = simple_rag_query(query)
    print("Response:", response)
    print(30*"-")

## Structured prompt

In [ ]:
def retrieve(query, top_k=2):
    query_embedding = retriever_model.encode(query, convert_to_tensor=True)
    query_embedding_np = query_embedding.cpu().numpy()
    query_embedding_np = query_embedding_np.reshape(1, -1)
    distances, top_k_indices = index.search(query_embedding_np, k=top_k)
    print('Top-k indices:', top_k_indices)

    # Fetch top-k relevant documents
    retrieved_docs = [corpus[idx] for idx in top_k_indices[0]]

    return retrieved_docs

def augment(query, retrieved_docs):
    # Create structured prompt
    prompt = """\
### VLOGA

Si strokovni asistent za podajanje informacij na podlagi specifične baze znanja. Tvoja naloga je, da uporabniku odgovoriš na vprašanje izključno z uporabo podanih "Virov".

### KONTEKST (VIRI)
{{context}}

### NAVODILA ZA ODGOVARJANJE
1. ODGOVARJAJ IZKLJUČNO NA PODLAGI VIROV: Ne uporabljaj svojega splošnega znanja o svetu. Če informacije ni v virih, to jasno povej.
2. CITIRANJE: Vsako trditev, ki jo povzameš po viru, označi z referenco v oglatih oklepajih, npr. [Vir 1] ali [naslov_dokumenta.pdf].
3. JEZIK IN TON: Odgovarjaj v slovenščini. Bodi vljuden, strokoven in jedrnat.
4. NERAZPOLOŽLJIVOST: Če odgovor na vprašanje ni vsebovan v podanih virih, odgovori natanko takole: "Žal v razpoložljivi bazi podatkov ne najdem informacij, ki bi odgovorile na vaše vprašanje."

### UPORABNIKOVO VPRAŠANJE
{{query}}\
"""

    # Add source numbers to the documents
    context = [f"Vir {i+1}: {document}" for i, document in enumerate(retrieved_docs)]
    
    # Add context and query to the prompt
    prompt = prompt.replace("{{context}}", "\n\n".join(context))
    prompt = prompt.replace("{{query}}", query)

    return prompt

def structured_rag_query(query):
    retrieved_docs = retrieve(query)
    prompt = augment(query, retrieved_docs)
    print("Augmented prompt:", prompt)

    return gams3_generate(prompt) 


In [ ]:
for query in prompts:
    print(30*"-")
    print("Query:", query)
    response = structured_rag_query(query)
    print("Response:", response)
    print(30*"-")

### Prompt that requires more sources

Adding more documents to the prompt is beneficial for two reasons:
1. The retriever does not work perfectly. If we add more documents to the prompt, we increase the probability of including the correct document
2. Sometimes the query can be more complicated, requiring the context from multiple documents.

We will explore the second option in our last example.

In [ ]:
query = "Kaj je projekt Triglav in kako postavim delujočo različico v lokalnem okolju?"

print(30*"-")
print("Query:", query)
response = structured_rag_query(query)
print("Response:", response)
print(30*"-")